# Summary Memory

> **Source:** `repo1/conversation_memory.py` → `demo_summary_memory()`


## Imports


In [ ]:
from langchain_openai import ChatOpenAI
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import (
    HumanMessage,
    AIMessage,
    SystemMessage,
    trim_messages,
)
from langchain_core.chat_history import (
    InMemoryChatMessageHistory,
    BaseChatMessageHistory,
)
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.output_parsers import StrOutputParser
from typing import Dict
from dotenv import load_dotenv


## Configuration & Setup


In [ ]:
load_dotenv()

llm = init_chat_model("gpt-4o-mini")


## Helper Function: `exercise_persistent_memory`


In [ ]:
def exercise_persistent_memory():
    """
    EXERCISE: Build a chatbot with:
    1. Persistent memory (SQLite)
    2. Automatic summarization after 10 messages
    3. User preference tracking

    Hint: Combine RunnableWithMessageHistory with SQLChatMessageHistory
    """

    print("=" * 60)
    print("EXERCISE: Persistent Memory Chatbot")
    print("=" * 60)

    from langchain_community.chat_message_histories import SQLChatMessageHistory
    import os

    # Use SQLite for persistence
    db_path = "./chat_history.db"

    def get_session_history(session_id: str) -> BaseChatMessageHistory:
        return SQLChatMessageHistory(
            session_id=session_id, connection=f"sqlite:///{db_path}"
        )

    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", "You are a helpful assistant. Remember user preferences."),
            MessagesPlaceholder(variable_name="history"),
            ("human", "{input}"),
        ]
    )
    chain = prompt | llm | StrOutputParser()

    chain_with_history = RunnableWithMessageHistory(
        chain,
        get_session_history,
        input_messages_key="input",
        history_messages_key="history",
    )

    config = {"configurable": {"session_id": "persistent_user"}}

    print("\nPersistent memory chatbot:")
    print("(Messages saved to SQLite database)\n")

    # Test conversation
    test_messages = [
        "Remember that I prefer dark mode themes",
        "What theme do I prefer?",
    ]

    for msg in test_messages:
        print(f"User: {msg}")
        response = chain_with_history.invoke({"input": msg}, config=config)
        print(f"AI: {response}\n")

    print(f"Database created: {db_path}")
    print("Messages persist across restarts!")

    # Cleanup for demo
    if os.path.exists(db_path):
        os.remove(db_path)


## Helper Function: `exercise_persistent_memory_proof`


In [ ]:
def exercise_persistent_memory_proof():
    """
    EXERCISE: Build a chatbot with:
    1. Persistent memory (SQLite)
    2. Proof that messages survive across separate chain instances
    3. User preference tracking

    Key idea: We create the chain TWICE to simulate two separate program runs.
    The second run reads from the same SQLite DB and recalls what the first run stored.
    """

    print("=" * 60)
    print("EXERCISE: Persistent Memory Chatbot")
    print("=" * 60)

    from langchain_community.chat_message_histories import SQLChatMessageHistory
    import sqlite3
    import os

    db_path = "./chat_history.db"
    connection_string = f"sqlite:///{db_path}"
    session_id = "persistent_user"

    # Clean slate
    if os.path.exists(db_path):
        os.remove(db_path)

    # --- Helper: build a fresh chain (simulates a new program run) ---
    def build_chain():
        llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

        def get_session_history(sid: str) -> BaseChatMessageHistory:
            return SQLChatMessageHistory(
                session_id=sid,
                connection=connection_string,
            )

        prompt = ChatPromptTemplate.from_messages(
            [
                (
                    "system",
                    "You are a helpful assistant. Remember user preferences and facts.",
                ),
                MessagesPlaceholder(variable_name="history"),
                ("human", "{input}"),
            ]
        )

        chain = prompt | llm | StrOutputParser()

        return RunnableWithMessageHistory(
            chain,
            get_session_history,
            input_messages_key="input",
            history_messages_key="history",
        )

    config = {"configurable": {"session_id": session_id}}

    # =====================================================
    # RUN 1 -- Store preferences (simulates first session)
    # =====================================================
    print("\n--- RUN 1: Storing preferences ---\n")

    chain_v1 = build_chain()

    run1_messages = [
        "My name is Paulo. I prefer dark mode themes and Python over JavaScript.",
        "I also like my responses concise -- no fluff.",
    ]

    for msg in run1_messages:
        print(f"User: {msg}")
        response = chain_v1.invoke({"input": msg}, config=config)
        print(f"AI:   {response}\n")

    # Throw away the chain object entirely -- no in-memory state survives
    del chain_v1

    # =====================================================
    # PROOF: Inspect the raw SQLite database
    # =====================================================
    print("--- DATABASE PROOF ---\n")
    print(f"Database file exists: {os.path.exists(db_path)}")
    print(f"Database size: {os.path.getsize(db_path)} bytes\n")

    conn = sqlite3.connect(db_path)
    cursor = conn.execute("SELECT * FROM message_store ORDER BY rowid")
    rows = cursor.fetchall()
    print(f"Total messages stored in DB: {len(rows)}\n")

    for i, row in enumerate(rows):
        print(
            f"  Row {i + 1}: session={row[0] if len(row) > 0 else 'N/A'}, "
            f"message (first 80 chars): {str(row[1])[:80] if len(row) > 1 else 'N/A'}..."
        )
    conn.close()

    # =====================================================
    # RUN 2 -- Brand new chain, same DB (simulates restart)
    # =====================================================
    print("\n--- RUN 2: Fresh chain, testing recall ---\n")

    chain_v2 = build_chain()

    recall_questions = [
        "What's my name?",
        "What theme do I prefer?",
        "What programming language do I prefer?",
        "How do I like my responses?",
    ]

    for msg in recall_questions:
        print(f"User: {msg}")
        response = chain_v2.invoke({"input": msg}, config=config)
        print(f"AI:   {response}\n")

    del chain_v2

    # =====================================================
    # FINAL: Show total messages accumulated
    # =====================================================
    print("--- FINAL DATABASE STATE ---\n")
    conn = sqlite3.connect(db_path)
    cursor = conn.execute("SELECT COUNT(*) FROM message_store")
    count = cursor.fetchone()[0]
    conn.close()

    print(f"Total messages in DB after both runs: {count}")
    print("Key insight: The second chain had ZERO in-memory history.")
    print("Everything was loaded from SQLite -- true persistence!")

    # Cleanup
    if os.path.exists(db_path):
        os.remove(db_path)


## Demo: Summary Memory


In [ ]:
def demo_summary_memory():
    """End-to-end summary memory: auto-summarize old messages, keep recent ones verbatim."""

    print("=" * 60)
    print("SUMMARY MEMORY")
    print("Summarize older messages to save tokens")
    print("=" * 60)

    # --- Setup ---
    summary_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    chat_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

    # The conversation prompt: summary of old context + recent messages
    chat_prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "You are a helpful assistant. Be concise.\n\n"
                "Summary of earlier conversation:\n{summary}",
            ),
            MessagesPlaceholder(variable_name="recent_messages"),
            ("human", "{input}"),
        ]
    )

    chat_chain = chat_prompt | chat_llm | StrOutputParser()

    # The summarization prompt: compress messages into a running summary
    summarize_prompt = ChatPromptTemplate.from_template(
        "Condense the current summary and new messages into a single updated summary "
        "(2-3 sentences). Preserve all key facts about the user.\n\n"
        "Current summary:\n{current_summary}\n\n"
        "New messages:\n{new_messages}\n\n"
        "Updated summary:"
    )

    summarize_chain = summarize_prompt | summary_llm | StrOutputParser()

    # --- State ---
    running_summary = ""  # starts empty
    recent_messages = []  # full message objects
    MAX_RECENT = 4  # keep last 4 messages (2 exchanges) before summarizing

    # --- Conversation ---
    exchanges = [
        "My name is Paulo and I'm from Seattle",
        "I work as an AI engineer building RAG systems",
        "I have 2 cats named Luna and Milo",
        "I'm building a LangChain course for Udemy",
        "What do you know about me? List everything.",
    ]

    print(f"\nConfig: keep last {MAX_RECENT} messages, summarize the rest\n")

    for user_input in exchanges:
        print(f"User: {user_input}")

        # 1. Call the LLM with summary + recent messages + new input
        response = chat_chain.invoke(
            {
                "summary": (
                    running_summary if running_summary else "No prior conversation."
                ),
                "recent_messages": recent_messages,
                "input": user_input,
            }
        )
        print(f"AI: {response}")

        # 2. Add this exchange to recent messages
        recent_messages.append(HumanMessage(content=user_input))
        recent_messages.append(AIMessage(content=response))

        # 3. If recent messages exceed limit, summarize the oldest ones
        if len(recent_messages) > MAX_RECENT:
            # Take the oldest messages that will be summarized away
            messages_to_summarize = recent_messages[:-MAX_RECENT]
            formatted = "\n".join(
                f"{'Human' if isinstance(m, HumanMessage) else 'AI'}: {m.content}"
                for m in messages_to_summarize
            )

            # Update the running summary
            running_summary = summarize_chain.invoke(
                {
                    "current_summary": (
                        running_summary if running_summary else "None yet."
                    ),
                    "new_messages": formatted,
                }
            )

            # Keep only the most recent messages
            recent_messages = recent_messages[-MAX_RECENT:]

            print(
                f"  >>> Summarized! Compressed {len(messages_to_summarize)} old messages"
            )
            print(f"  >>> Summary: {running_summary}")
            print(f"  >>> Recent buffer: {len(recent_messages)} messages")
        print()

    # --- Final state ---
    print("=" * 60)
    print("FINAL MEMORY STATE")
    print("=" * 60)
    print(f"\nRunning summary (compressed old context):\n  {running_summary}")
    print(f"\nRecent messages kept verbatim ({len(recent_messages)}):")
    for msg in recent_messages:
        role = "Human" if isinstance(msg, HumanMessage) else "AI"
        print(f"  {role}: {msg.content[:80]}")
    print("\nKey insight: ALL facts preserved (name, city, job, cats, course)")
    print("But token cost stays bounded -- old messages are compressed, not deleted!")


## Execute


In [ ]:
demo_summary_memory()
